<a href="https://colab.research.google.com/github/tadeu-andre/Mestrado/blob/main/Mestrado_MVP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CÉLULA 1: Instalação de Dependências
!pip install fastapi uvicorn sqlalchemy pydantic python-multipart
!pip install streamlit plotly pandas openpyxl
!pip install python-docx jinja2 weasyprint
!pip install pyngrok  # Para expor a API publicamente
!pip install nest-asyncio  # Para rodar FastAPI no Colab

print("✅ Dependências instaladas com sucesso!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 851.1/851.1 kB 47.5 MB/s eta 0:00:00
✅ Dependências instaladas com sucesso!


In [2]:
# CÉLULA 2: Criar estrutura do projeto
import os

# Criar diretórios
os.makedirs('backend', exist_ok=True)
os.makedirs('backend/routers', exist_ok=True)
os.makedirs('frontend', exist_ok=True)
os.makedirs('templates', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Criar arquivos __init__.py
open('backend/__init__.py', 'a').close()
open('backend/routers/__init__.py', 'a').close()

print("✅ Estrutura de pastas criada:")
print("""
project/
├── backend/
│   ├── __init__.py
│   ├── models.py
│   ├── schemas.py
│   ├── crud.py
│   ├── database.py
│   ├── main.py
│   └── routers/
│       ├── __init__.py
│       ├── empresas.py
│       ├── projetos.py
│       ├── despesas.py
│       └── documentos.py
├── frontend/
│   └── app.py
├── templates/
│   └── plano_trabalho.docx
└── data/
    └── database.db (SQLite)
""")

✅ Estrutura de pastas criada:

project/
├── backend/
│   ├── __init__.py
│   ├── models.py
│   ├── schemas.py
│   ├── crud.py
│   ├── database.py
│   ├── main.py
│   └── routers/
│       ├── __init__.py
│       ├── empresas.py
│       ├── projetos.py
│       ├── despesas.py
│       └── documentos.py
├── frontend/
│   └── app.py
├── templates/
│   └── plano_trabalho.docx
└── data/
    └── database.db (SQLite)



In [3]:
%%writefile backend/database.py
# CÉLULA 3: Configuração do Banco de Dados

from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# SQLite para simplicidade no Colab
SQLALCHEMY_DATABASE_URL = "sqlite:///./data/database.db"

engine = create_engine(
    SQLALCHEMY_DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

Base = declarative_base()

def get_db():
    """Dependency para obter sessão do banco"""
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

def init_db():
    """Inicializa o banco de dados criando todas as tabelas"""
    Base.metadata.create_all(bind=engine)
    print("✅ Banco de dados inicializado!")

Writing backend/database.py


In [11]:
%%writefile backend/models.py
# CÉLULA 4 (CORRIGIDA): Models SQLAlchemy

from sqlalchemy import Column, String, Date, DateTime, Integer, Text, ForeignKey, Enum, JSON, CheckConstraint, Numeric
from sqlalchemy.orm import relationship
from sqlalchemy.sql import func
from backend.database import Base
import enum
import uuid

# Enums
class StatusProjeto(str, enum.Enum):
    ATIVO = "ativo"
    CONCLUIDO = "concluido"
    CANCELADO = "cancelado"

class CategoriaDespesa(str, enum.Enum):
    PESSOAL = "Pessoal"
    MATERIAL = "Material"
    SERVICOS = "Serviços"
    EQUIPAMENTOS = "Equipamentos"
    OUTROS = "Outros"

class TipoDesembolso(str, enum.Enum):
    EMBRAPII = "embrapii"
    SEBRAE = "sebrae"
    BNDES = "bndes"
    CUSTOM = "custom"

# Models
class Empresa(Base):
    __tablename__ = "empresas"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    nome = Column(String(200), nullable=False, index=True)
    cnpj = Column(String(14), unique=True, nullable=False, index=True)
    email = Column(String(100))
    telefone = Column(String(20))
    created_at = Column(DateTime, server_default=func.now())
    updated_at = Column(DateTime, server_default=func.now(), onupdate=func.now())

    # Relacionamentos
    projetos = relationship("Projeto", back_populates="empresa", cascade="all, delete-orphan")

class Projeto(Base):
    __tablename__ = "projetos"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    empresa_id = Column(String(36), ForeignKey("empresas.id"), nullable=False)
    titulo = Column(String(300), nullable=False, index=True)
    descricao = Column(Text)
    valor_total = Column(Numeric(15, 2), default=0)
    data_inicio = Column(Date)
    data_fim = Column(Date)
    status = Column(Enum(StatusProjeto), default=StatusProjeto.ATIVO)
    created_at = Column(DateTime, server_default=func.now())
    updated_at = Column(DateTime, server_default=func.now(), onupdate=func.now())

    # Relacionamentos
    empresa = relationship("Empresa", back_populates="projetos")
    etapas = relationship("Etapa", back_populates="projeto", cascade="all, delete-orphan", order_by="Etapa.ordem")
    despesas = relationship("ItemDespesa", back_populates="projeto", cascade="all, delete-orphan")
    desembolsos = relationship("Desembolso", back_populates="projeto", cascade="all, delete-orphan")

class Etapa(Base):
    __tablename__ = "etapas"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    projeto_id = Column(String(36), ForeignKey("projetos.id"), nullable=False)
    nome = Column(String(200), nullable=False)
    descricao = Column(Text)
    ordem = Column(Integer, nullable=False)
    created_at = Column(DateTime, server_default=func.now())

    # Relacionamentos
    projeto = relationship("Projeto", back_populates="etapas")
    atividades = relationship("Atividade", back_populates="etapa", cascade="all, delete-orphan", order_by="Atividade.data_inicio")

class Atividade(Base):
    __tablename__ = "atividades"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    etapa_id = Column(String(36), ForeignKey("etapas.id"), nullable=False)
    nome = Column(String(200), nullable=False)
    descricao = Column(Text)
    data_inicio = Column(Date, nullable=False)
    data_fim = Column(Date, nullable=False)
    responsavel = Column(String(100))
    created_at = Column(DateTime, server_default=func.now())

    # Constraint: data_fim >= data_inicio
    __table_args__ = (
        CheckConstraint('data_fim >= data_inicio', name='check_datas_atividade'),
    )

    # Relacionamentos
    etapa = relationship("Etapa", back_populates="atividades")

class ItemDespesa(Base):
    __tablename__ = "despesas"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    projeto_id = Column(String(36), ForeignKey("projetos.id"), nullable=False)
    descricao = Column(String(300), nullable=False)
    valor = Column(Numeric(15, 2), nullable=False)
    categoria = Column(Enum(CategoriaDespesa), nullable=False)
    percentual_embrapii = Column(Numeric(5, 2), default=0)
    percentual_empresa = Column(Numeric(5, 2), default=0)
    percentual_iff = Column(Numeric(5, 2), default=0)
    created_at = Column(DateTime, server_default=func.now())

    # Constraint: soma dos percentuais = 100
    __table_args__ = (
        CheckConstraint(
            'percentual_embrapii + percentual_empresa + percentual_iff = 100',
            name='check_percentuais_soma_100'
        ),
    )

    # Relacionamentos
    projeto = relationship("Projeto", back_populates="despesas")

class DesembolsoRegra(Base):
    __tablename__ = "desembolso_regras"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    nome = Column(String(100), unique=True, nullable=False)
    descricao = Column(Text)
    tipo = Column(Enum(TipoDesembolso), nullable=False)
    parametros = Column(JSON, nullable=False)
    created_at = Column(DateTime, server_default=func.now())

class Desembolso(Base):
    __tablename__ = "desembolsos"

    id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))
    projeto_id = Column(String(36), ForeignKey("projetos.id"), nullable=False)
    desembolso_regra_id = Column(String(36), ForeignKey("desembolso_regras.id"))
    parcela = Column(Integer, nullable=False)
    valor = Column(Numeric(15, 2), nullable=False)
    data_prevista = Column(Date, nullable=False)
    percentual = Column(Numeric(5, 2))
    created_at = Column(DateTime, server_default=func.now())

    # Relacionamentos
    projeto = relationship("Projeto", back_populates="desembolsos")
    regra = relationship("DesembolsoRegra")

Overwriting backend/models.py


In [5]:
%%writefile backend/schemas.py
# CÉLULA 5: Schemas Pydantic para validação

from pydantic import BaseModel, Field, field_validator
from typing import Optional, List
from datetime import date, datetime
from decimal import Decimal
from enum import Enum

# Enums
class StatusProjetoEnum(str, Enum):
    ATIVO = "ativo"
    CONCLUIDO = "concluido"
    CANCELADO = "cancelado"

class CategoriaDespesaEnum(str, Enum):
    PESSOAL = "Pessoal"
    MATERIAL = "Material"
    SERVICOS = "Serviços"
    EQUIPAMENTOS = "Equipamentos"
    OUTROS = "Outros"

class TipoDesembolsoEnum(str, Enum):
    EMBRAPII = "embrapii"
    SEBRAE = "sebrae"
    BNDES = "bndes"
    CUSTOM = "custom"

# Empresa Schemas
class EmpresaBase(BaseModel):
    nome: str = Field(..., min_length=3, max_length=200)
    cnpj: str = Field(..., pattern=r'^\d{14}$')
    email: Optional[str] = Field(None, max_length=100)
    telefone: Optional[str] = Field(None, max_length=20)

class EmpresaCreate(EmpresaBase):
    pass

class EmpresaUpdate(BaseModel):
    nome: Optional[str] = Field(None, min_length=3, max_length=200)
    email: Optional[str] = Field(None, max_length=100)
    telefone: Optional[str] = Field(None, max_length=20)

class Empresa(EmpresaBase):
    id: str
    created_at: datetime
    updated_at: datetime

    class Config:
        from_attributes = True

# Projeto Schemas
class ProjetoBase(BaseModel):
    titulo: str = Field(..., min_length=5, max_length=300)
    descricao: Optional[str] = None
    valor_total: Optional[Decimal] = Field(default=0, ge=0)
    data_inicio: Optional[date] = None
    data_fim: Optional[date] = None
    status: StatusProjetoEnum = StatusProjetoEnum.ATIVO

class ProjetoCreate(ProjetoBase):
    empresa_id: str

class ProjetoUpdate(BaseModel):
    titulo: Optional[str] = Field(None, min_length=5, max_length=300)
    descricao: Optional[str] = None
    valor_total: Optional[Decimal] = Field(None, ge=0)
    data_inicio: Optional[date] = None
    data_fim: Optional[date] = None
    status: Optional[StatusProjetoEnum] = None

class Projeto(ProjetoBase):
    id: str
    empresa_id: str
    created_at: datetime
    updated_at: datetime

    class Config:
        from_attributes = True

# Etapa Schemas
class EtapaBase(BaseModel):
    nome: str = Field(..., min_length=3, max_length=200)
    descricao: Optional[str] = None
    ordem: int = Field(..., ge=1)

class EtapaCreate(EtapaBase):
    projeto_id: str

class EtapaUpdate(BaseModel):
    nome: Optional[str] = Field(None, min_length=3, max_length=200)
    descricao: Optional[str] = None
    ordem: Optional[int] = Field(None, ge=1)

class Etapa(EtapaBase):
    id: str
    projeto_id: str
    created_at: datetime

    class Config:
        from_attributes = True

# Atividade Schemas
class AtividadeBase(BaseModel):
    nome: str = Field(..., min_length=3, max_length=200)
    descricao: Optional[str] = None
    data_inicio: date
    data_fim: date
    responsavel: Optional[str] = Field(None, max_length=100)

    @field_validator('data_fim')
    @classmethod
    def validar_datas(cls, v, info):
        if 'data_inicio' in info.data and v < info.data['data_inicio']:
            raise ValueError('data_fim deve ser maior ou igual a data_inicio')
        return v

class AtividadeCreate(AtividadeBase):
    etapa_id: str

class AtividadeUpdate(BaseModel):
    nome: Optional[str] = Field(None, min_length=3, max_length=200)
    descricao: Optional[str] = None
    data_inicio: Optional[date] = None
    data_fim: Optional[date] = None
    responsavel: Optional[str] = Field(None, max_length=100)

class Atividade(AtividadeBase):
    id: str
    etapa_id: str
    created_at: datetime

    class Config:
        from_attributes = True

# ItemDespesa Schemas
class ItemDespesaBase(BaseModel):
    descricao: str = Field(..., min_length=3, max_length=300)
    valor: Decimal = Field(..., gt=0)
    categoria: CategoriaDespesaEnum
    percentual_embrapii: Decimal = Field(default=0, ge=0, le=100)
    percentual_empresa: Decimal = Field(default=0, ge=0, le=100)
    percentual_iff: Decimal = Field(default=0, ge=0, le=100)

    @field_validator('percentual_iff')
    @classmethod
    def validar_percentuais(cls, v, info):
        total = (
            info.data.get('percentual_embrapii', 0) +
            info.data.get('percentual_empresa', 0) +
            v
        )
        if abs(total - 100) > 0.01:  # Tolerância para arredondamento
            raise ValueError('A soma dos percentuais deve ser exatamente 100%')
        return v

class ItemDespesaCreate(ItemDespesaBase):
    projeto_id: str

class ItemDespesaUpdate(BaseModel):
    descricao: Optional[str] = Field(None, min_length=3, max_length=300)
    valor: Optional[Decimal] = Field(None, gt=0)
    categoria: Optional[CategoriaDespesaEnum] = None
    percentual_embrapii: Optional[Decimal] = Field(None, ge=0, le=100)
    percentual_empresa: Optional[Decimal] = Field(None, ge=0, le=100)
    percentual_iff: Optional[Decimal] = Field(None, ge=0, le=100)

class ItemDespesa(ItemDespesaBase):
    id: str
    projeto_id: str
    created_at: datetime

    class Config:
        from_attributes = True

# DesembolsoRegra Schemas
class DesembolsoRegraBase(BaseModel):
    nome: str = Field(..., min_length=3, max_length=100)
    descricao: Optional[str] = None
    tipo: TipoDesembolsoEnum
    parametros: dict

class DesembolsoRegraCreate(DesembolsoRegraBase):
    pass

class DesembolsoRegra(DesembolsoRegraBase):
    id: str
    created_at: datetime

    class Config:
        from_attributes = True

# Desembolso Schemas
class DesembolsoBase(BaseModel):
    parcela: int = Field(..., ge=1)
    valor: Decimal = Field(..., gt=0)
    data_prevista: date
    percentual: Optional[Decimal] = Field(None, ge=0, le=100)

class DesembolsoCreate(DesembolsoBase):
    projeto_id: str
    desembolso_regra_id: Optional[str] = None

class Desembolso(DesembolsoBase):
    id: str
    projeto_id: str
    desembolso_regra_id: Optional[str]
    created_at: datetime

    class Config:
        from_attributes = True

# Schemas de Resposta Completa
class EtapaComAtividades(Etapa):
    atividades: List[Atividade] = []

class ProjetoCompleto(Projeto):
    empresa: Empresa
    etapas: List[EtapaComAtividades] = []
    despesas: List[ItemDespesa] = []
    desembolsos: List[Desembolso] = []

# Schemas para Relatórios
class OrcamentoConsolidado(BaseModel):
    total_geral: Decimal
    por_categoria: dict
    por_fonte: dict

class CronogramaFinanceiro(BaseModel):
    meses: List[str]
    valores: List[Decimal]
    acumulado: List[Decimal]

Writing backend/schemas.py


In [6]:
%%writefile backend/crud.py
# CÉLULA 6: Operações CRUD

from sqlalchemy.orm import Session, joinedload
from sqlalchemy import func, extract
from typing import List, Optional
from datetime import date, datetime
from decimal import Decimal
import backend.models as models
import backend.schemas as schemas

# ==================== EMPRESA ====================

def criar_empresa(db: Session, empresa: schemas.EmpresaCreate):
    """Cria nova empresa"""
    db_empresa = models.Empresa(**empresa.model_dump())
    db.add(db_empresa)
    db.commit()
    db.refresh(db_empresa)
    return db_empresa

def listar_empresas(db: Session, skip: int = 0, limit: int = 100):
    """Lista todas as empresas"""
    return db.query(models.Empresa).offset(skip).limit(limit).all()

def obter_empresa(db: Session, empresa_id: str):
    """Obtém empresa por ID"""
    return db.query(models.Empresa).filter(models.Empresa.id == empresa_id).first()

def obter_empresa_por_cnpj(db: Session, cnpj: str):
    """Obtém empresa por CNPJ"""
    return db.query(models.Empresa).filter(models.Empresa.cnpj == cnpj).first()

def atualizar_empresa(db: Session, empresa_id: str, empresa: schemas.EmpresaUpdate):
    """Atualiza empresa existente"""
    db_empresa = obter_empresa(db, empresa_id)
    if not db_empresa:
        return None

    update_data = empresa.model_dump(exclude_unset=True)
    for key, value in update_data.items():
        setattr(db_empresa, key, value)

    db.commit()
    db.refresh(db_empresa)
    return db_empresa

def deletar_empresa(db: Session, empresa_id: str):
    """Deleta empresa"""
    db_empresa = obter_empresa(db, empresa_id)
    if not db_empresa:
        return False

    db.delete(db_empresa)
    db.commit()
    return True

# ==================== PROJETO ====================

def criar_projeto(db: Session, projeto: schemas.ProjetoCreate):
    """Cria novo projeto"""
    db_projeto = models.Projeto(**projeto.model_dump())
    db.add(db_projeto)
    db.commit()
    db.refresh(db_projeto)
    return db_projeto

def listar_projetos(db: Session, skip: int = 0, limit: int = 100, empresa_id: Optional[str] = None):
    """Lista projetos, opcionalmente filtrados por empresa"""
    query = db.query(models.Projeto)
    if empresa_id:
        query = query.filter(models.Projeto.empresa_id == empresa_id)
    return query.offset(skip).limit(limit).all()

def obter_projeto(db: Session, projeto_id: str):
    """Obtém projeto por ID"""
    return db.query(models.Projeto).filter(models.Projeto.id == projeto_id).first()

def obter_projeto_completo(db: Session, projeto_id: str):
    """Obtém projeto com todos os relacionamentos"""
    return db.query(models.Projeto)\
        .options(
            joinedload(models.Projeto.empresa),
            joinedload(models.Projeto.etapas).joinedload(models.Etapa.atividades),
            joinedload(models.Projeto.despesas),
            joinedload(models.Projeto.desembolsos)
        )\
        .filter(models.Projeto.id == projeto_id)\
        .first()

def atualizar_projeto(db: Session, projeto_id: str, projeto: schemas.ProjetoUpdate):
    """Atualiza projeto existente"""
    db_projeto = obter_projeto(db, projeto_id)
    if not db_projeto:
        return None

    update_data = projeto.model_dump(exclude_unset=True)
    for key, value in update_data.items():
        setattr(db_projeto, key, value)

    db.commit()
    db.refresh(db_projeto)
    return db_projeto

def deletar_projeto(db: Session, projeto_id: str):
    """Deleta projeto"""
    db_projeto = obter_projeto(db, projeto_id)
    if not db_projeto:
        return False

    db.delete(db_projeto)
    db.commit()
    return True

def recalcular_datas_projeto(db: Session, projeto_id: str):
    """Recalcula data_inicio e data_fim do projeto baseado nas atividades"""
    atividades = db.query(models.Atividade)\
        .join(models.Etapa)\
        .filter(models.Etapa.projeto_id == projeto_id)\
        .all()

    if not atividades:
        return None

    data_inicio = min(a.data_inicio for a in atividades)
    data_fim = max(a.data_fim for a in atividades)

    db_projeto = obter_projeto(db, projeto_id)
    db_projeto.data_inicio = data_inicio
    db_projeto.data_fim = data_fim

    db.commit()
    db.refresh(db_projeto)
    return db_projeto

def recalcular_valor_projeto(db: Session, projeto_id: str):
    """Recalcula valor_total do projeto baseado nas despesas"""
    total = db.query(func.sum(models.ItemDespesa.valor))\
        .filter(models.ItemDespesa.projeto_id == projeto_id)\
        .scalar() or 0

    db_projeto = obter_projeto(db, projeto_id)
    db_projeto.valor_total = total

    db.commit()
    db.refresh(db_projeto)
    return db_projeto

# ==================== ETAPA ====================

def criar_etapa(db: Session, etapa: schemas.EtapaCreate):
    """Cria nova etapa"""
    db_etapa = models.Etapa(**etapa.model_dump())
    db.add(db_etapa)
    db.commit()
    db.refresh(db_etapa)
    return db_etapa

def listar_etapas(db: Session, projeto_id: str):
    """Lista etapas de um projeto"""
    return db.query(models.Etapa)\
        .filter(models.Etapa.projeto_id == projeto_id)\
        .order_by(models.Etapa.ordem)\
        .all()

def obter_etapa(db: Session, etapa_id: str):
    """Obtém etapa por ID"""
    return db.query(models.Etapa).filter(models.Etapa.id == etapa_id).first()

def atualizar_etapa(db: Session, etapa_id: str, etapa: schemas.EtapaUpdate):
    """Atualiza etapa existente"""
    db_etapa = obter_etapa(db, etapa_id)
    if not db_etapa:
        return None

    update_data = etapa.model_dump(exclude_unset=True)
    for key, value in update_data.items():
        setattr(db_etapa, key, value)

    db.commit()
    db.refresh(db_etapa)
    return db_etapa

def deletar_etapa(db: Session, etapa_id: str):
    """Deleta etapa"""
    db_etapa = obter_etapa(db, etapa_id)
    if not db_etapa:
        return False

    db.delete(db_etapa)
    db.commit()
    return True

# ==================== ATIVIDADE ====================

def criar_atividade(db: Session, atividade: schemas.AtividadeCreate):
    """Cria nova atividade"""
    db_atividade = models.Atividade(**atividade.model_dump())
    db.add(db_atividade)
    db.commit()
    db.refresh(db_atividade)

    # Recalcula datas do projeto
    etapa = obter_etapa(db, atividade.etapa_id)
    recalcular_datas_projeto(db, etapa.projeto_id)

    return db_atividade

def listar_atividades(db: Session, etapa_id: str):
    """Lista atividades de uma etapa"""
    return db.query(models.Atividade)\
        .filter(models.Atividade.etapa_id == etapa_id)\
        .order_by(models.Atividade.data_inicio)\
        .all()

def obter_atividade(db: Session, atividade_id: str):
    """Obtém atividade por ID"""
    return db.query(models.Atividade).filter(models.Atividade.id == atividade_id).first()

def atualizar_atividade(db: Session, atividade_id: str, atividade: schemas.AtividadeUpdate):
    """Atualiza atividade existente"""
    db_atividade = obter_atividade(db, atividade_id)
    if not db_atividade:
        return None

    update_data = atividade.model_dump(exclude_unset=True)
    for key, value in update_data.items():
        setattr(db_atividade, key, value)

    db.commit()
    db.refresh(db_atividade)

    # Recalcula datas do projeto
    etapa = obter_etapa(db, db_atividade.etapa_id)
    recalcular_datas_projeto(db, etapa.projeto_id)

    return db_atividade

def deletar_atividade(db: Session, atividade_id: str):
    """Deleta atividade"""
    db_atividade = obter_atividade(db, atividade_id)
    if not db_atividade:
        return False

    etapa = obter_etapa(db, db_atividade.etapa_id)
    projeto_id = etapa.projeto_id

    db.delete(db_atividade)
    db.commit()

    # Recalcula datas do projeto
    recalcular_datas_projeto(db, projeto_id)

    return True

# ==================== DESPESA ====================

def criar_despesa(db: Session, despesa: schemas.ItemDespesaCreate):
    """Cria nova despesa"""
    db_despesa = models.ItemDespesa(**despesa.model_dump())
    db.add(db_despesa)
    db.commit()
    db.refresh(db_despesa)

    # Recalcula valor total do projeto
    recalcular_valor_projeto(db, despesa.projeto_id)

    return db_despesa

def listar_despesas(db: Session, projeto_id: str):
    """Lista despesas de um projeto"""
    return db.query(models.ItemDespesa)\
        .filter(models.ItemDespesa.projeto_id == projeto_id)\
        .all()

def obter_despesa(db: Session, despesa_id: str):
    """Obtém despesa por ID"""
    return db.query(models.ItemDespesa).filter(models.ItemDespesa.id == despesa_id).first()

def atualizar_despesa(db: Session, despesa_id: str, despesa: schemas.ItemDespesaUpdate):
    """Atualiza despesa existente"""
    db_despesa = obter_despesa(db, despesa_id)
    if not db_despesa:
        return None

    update_data = despesa.model_dump(exclude_unset=True)
    for key, value in update_data.items():
        setattr(db_despesa, key, value)

    db.commit()
    db.refresh(db_despesa)

    # Recalcula valor total do projeto
    recalcular_valor_projeto(db, db_despesa.projeto_id)

    return db_despesa

def deletar_despesa(db: Session, despesa_id: str):
    """Deleta despesa"""
    db_despesa = obter_despesa(db, despesa_id)
    if not db_despesa:
        return False

    projeto_id = db_despesa.projeto_id

    db.delete(db_despesa)
    db.commit()

    # Recalcula valor total do projeto
    recalcular_valor_projeto(db, projeto_id)

    return True

# ==================== ORÇAMENTO CONSOLIDADO ====================

def obter_orcamento_consolidado(db: Session, projeto_id: str):
    """Gera orçamento consolidado por categoria e fonte"""
    despesas = listar_despesas(db, projeto_id)

    if not despesas:
        return {
            "total_geral": 0,
            "por_categoria": {},
            "por_fonte": {
                "EMBRAPII": 0,
                "Empresa": 0,
                "IFF": 0
            }
        }

    # Consolidar por categoria
    por_categoria = {}
    for despesa in despesas:
        cat = despesa.categoria.value
        if cat not in por_categoria:
            por_categoria[cat] = 0
        por_categoria[cat] += float(despesa.valor)

    # Consolidar por fonte
    por_fonte = {
        "EMBRAPII": 0,
        "Empresa": 0,
        "IFF": 0
    }

    for despesa in despesas:
        valor = float(despesa.valor)
        por_fonte["EMBRAPII"] += valor * float(despesa.percentual_embrapii) / 100
        por_fonte["Empresa"] += valor * float(despesa.percentual_empresa) / 100
        por_fonte["IFF"] += valor * float(despesa.percentual_iff) / 100

    total_geral = sum(float(d.valor) for d in despesas)

    return {
        "total_geral": round(total_geral, 2),
        "por_categoria": {k: round(v, 2) for k, v in por_categoria.items()},
        "por_fonte": {k: round(v, 2) for k, v in por_fonte.items()}
    }

# ==================== CRONOGRAMA FINANCEIRO ====================

def obter_cronograma_financeiro(db: Session, projeto_id: str):
    """Gera cronograma financeiro mensal"""
    projeto = obter_projeto_completo(db, projeto_id)

    if not projeto or not projeto.data_inicio or not projeto.data_fim:
        return {"meses": [], "valores": [], "acumulado": []}

    # Gerar lista de meses
    meses = []
    data_atual = projeto.data_inicio.replace(day=1)
    data_fim = projeto.data_fim.replace(day=1)

    while data_atual <= data_fim:
        meses.append(data_atual.strftime("%Y-%m"))
        if data_atual.month == 12:
            data_atual = data_atual.replace(year=data_atual.year + 1, month=1)
        else:
            data_atual = data_atual.replace(month=data_atual.month + 1)

    # Distribuir despesas por mês
    valores = [0.0] * len(meses)

    for etapa in projeto.etapas:
        for atividade in etapa.atividades:
            # Calcular duração da atividade em dias
            duracao = (atividade.data_fim - atividade.data_inicio).days + 1

            # Encontrar despesas relacionadas (simplificado: todas as despesas do projeto)
            # Em produção, você poderia vincular despesas específicas a atividades
            valor_atividade = float(projeto.valor_total) / len([a for e in projeto.etapas for a in e.atividades])
            valor_diario = valor_atividade / duracao if duracao > 0 else 0

            # Distribuir pelos meses
            data_atual = atividade.data_inicio
            while data_atual <= atividade.data_fim:
                mes_str = data_atual.strftime("%Y-%m")
                if mes_str in meses:
                    idx = meses.index(mes_str)
                    valores[idx] += valor_diario

                # Avançar para o próximo dia
                from datetime import timedelta
                data_atual += timedelta(days=1)

    # Calcular acumulado
    acumulado = []
    soma = 0
    for valor in valores:
        soma += valor
        acumulado.append(round(soma, 2))

    return {
        "meses": meses,
        "valores": [round(v, 2) for v in valores],
        "acumulado": acumulado
    }

# ==================== DESEMBOLSO ====================

def criar_desembolso_regra(db: Session, regra: schemas.DesembolsoRegraCreate):
    """Cria novo modelo de desembolso"""
    db_regra = models.DesembolsoRegra(**regra.model_dump())
    db.add(db_regra)
    db.commit()
    db.refresh(db_regra)
    return db_regra

def listar_desembolso_regras(db: Session):
    """Lista todos os modelos de desembolso"""
    return db.query(models.DesembolsoRegra).all()

def obter_desembolso_regra(db: Session, regra_id: str):
    """Obtém modelo de desembolso por ID"""
    return db.query(models.DesembolsoRegra).filter(models.DesembolsoRegra.id == regra_id).first()

def aplicar_desembolso(db: Session, projeto_id: str, regra_id: str):
    """Aplica modelo de desembolso ao projeto"""
    projeto = obter_projeto(db, projeto_id)
    regra = obter_desembolso_regra(db, regra_id)

    if not projeto or not regra:
        return None

    if not projeto.data_inicio or not projeto.data_fim:
        return None

    # Limpar desembolsos anteriores
    db.query(models.Desembolso).filter(models.Desembolso.projeto_id == projeto_id).delete()

    # Aplicar nova regra
    parametros = regra.parametros
    distribuicao = parametros.get('distribuicao', [])

    from datetime import timedelta
    duracao_total = (projeto.data_fim - projeto.data_inicio).days

    desembolsos_criados = []

    for item in distribuicao:
        parcela = item['parcela']
        percentual = item['percentual']
        valor = float(projeto.valor_total) * percentual / 100

        # Calcular data baseada no momento
        momento = item.get('momento', 'inicio')
        if momento == 'inicio':
            data_prevista = projeto.data_inicio
        elif momento == 'meio':
            data_prevista = projeto.data_inicio + timedelta(days=duracao_total // 2)
        elif momento == 'fim':
            data_prevista = projeto.data_fim
        else:
            # Para modelos com trimestres
            trimestre = item.get('trimestre', 1)
            dias_trimestre = duracao_total // 4
            data_prevista = projeto.data_inicio + timedelta(days=dias_trimestre * (trimestre - 1))

        db_desembolso = models.Desembolso(
            projeto_id=projeto_id,
            desembolso_regra_id=regra_id,
            parcela=parcela,
            valor=valor,
            data_prevista=data_prevista,
            percentual=percentual
        )
        db.add(db_desembolso)
        desembolsos_criados.append(db_desembolso)

    db.commit()

    for d in desembolsos_criados:
        db.refresh(d)

    return desembolsos_criados

def listar_desembolsos(db: Session, projeto_id: str):
    """Lista desembolsos de um projeto"""
    return db.query(models.Desembolso)\
        .filter(models.Desembolso.projeto_id == projeto_id)\
        .order_by(models.Desembolso.parcela)\
        .all()

def comparar_desembolso_cronograma(db: Session, projeto_id: str):
    """Compara desembolso planejado com cronograma financeiro"""
    desembolsos = listar_desembolsos(db, projeto_id)
    cronograma = obter_cronograma_financeiro(db, projeto_id)

    total_desembolso = sum(float(d.valor) for d in desembolsos)
    total_cronograma = sum(cronograma['valores'])

    divergencia = total_desembolso - total_cronograma

    alertas = []
    if abs(divergencia) > 0.01:
        alertas.append({
            "tipo": "divergencia_total",
            "mensagem": f"Divergência de R$ {abs(divergencia):.2f} entre desembolso e cronograma",
            "severidade": "alta" if abs(divergencia) > total_cronograma * 0.1 else "media"
        })

    return {
        "total_desembolso": round(total_desembolso, 2),
        "total_cronograma": round(total_cronograma, 2),
        "divergencia": round(divergencia, 2),
        "alertas": alertas
    }

Writing backend/crud.py


In [7]:
%%writefile backend/main.py
# CÉLULA 7: Aplicação FastAPI Principal

from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from sqlalchemy.orm import Session
from typing import List, Optional
import backend.models as models
import backend.schemas as schemas
import backend.crud as crud
from backend.database import get_db, init_db, engine

# Criar aplicação FastAPI
app = FastAPI(
    title="Sistema de Gestão de Projetos P&D",
    description="API para gerenciamento de projetos de pesquisa e desenvolvimento",
    version="1.0.0"
)

# Configurar CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Inicializar banco de dados na inicialização
@app.on_event("startup")
def startup_event():
    init_db()
    print("✅ API inicializada e banco de dados criado!")

# ==================== ROTAS DE EMPRESA ====================

@app.post("/api/empresas", response_model=schemas.Empresa, status_code=status.HTTP_201_CREATED, tags=["Empresas"])
def criar_empresa_endpoint(empresa: schemas.EmpresaCreate, db: Session = Depends(get_db)):
    """Cria uma nova empresa"""
    # Verificar se CNPJ já existe
    db_empresa = crud.obter_empresa_por_cnpj(db, empresa.cnpj)
    if db_empresa:
        raise HTTPException(status_code=400, detail="CNPJ já cadastrado")
    return crud.criar_empresa(db, empresa)

@app.get("/api/empresas", response_model=List[schemas.Empresa], tags=["Empresas"])
def listar_empresas_endpoint(skip: int = 0, limit: int = 100, db: Session = Depends(get_db)):
    """Lista todas as empresas"""
    return crud.listar_empresas(db, skip, limit)

@app.get("/api/empresas/{empresa_id}", response_model=schemas.Empresa, tags=["Empresas"])
def obter_empresa_endpoint(empresa_id: str, db: Session = Depends(get_db)):
    """Obtém uma empresa específica"""
    db_empresa = crud.obter_empresa(db, empresa_id)
    if not db_empresa:
        raise HTTPException(status_code=404, detail="Empresa não encontrada")
    return db_empresa

@app.put("/api/empresas/{empresa_id}", response_model=schemas.Empresa, tags=["Empresas"])
def atualizar_empresa_endpoint(empresa_id: str, empresa: schemas.EmpresaUpdate, db: Session = Depends(get_db)):
    """Atualiza uma empresa"""
    db_empresa = crud.atualizar_empresa(db, empresa_id, empresa)
    if not db_empresa:
        raise HTTPException(status_code=404, detail="Empresa não encontrada")
    return db_empresa

@app.delete("/api/empresas/{empresa_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["Empresas"])
def deletar_empresa_endpoint(empresa_id: str, db: Session = Depends(get_db)):
    """Deleta uma empresa"""
    success = crud.deletar_empresa(db, empresa_id)
    if not success:
        raise HTTPException(status_code=404, detail="Empresa não encontrada")

# ==================== ROTAS DE PROJETO ====================

@app.post("/api/projetos", response_model=schemas.Projeto, status_code=status.HTTP_201_CREATED, tags=["Projetos"])
def criar_projeto_endpoint(projeto: schemas.ProjetoCreate, db: Session = Depends(get_db)):
    """Cria um novo projeto"""
    # Verificar se empresa existe
    empresa = crud.obter_empresa(db, projeto.empresa_id)
    if not empresa:
        raise HTTPException(status_code=404, detail="Empresa não encontrada")
    return crud.criar_projeto(db, projeto)

@app.get("/api/projetos", response_model=List[schemas.Projeto], tags=["Projetos"])
def listar_projetos_endpoint(
    skip: int = 0,
    limit: int = 100,
    empresa_id: Optional[str] = None,
    db: Session = Depends(get_db)
):
    """Lista todos os projetos"""
    return crud.listar_projetos(db, skip, limit, empresa_id)

@app.get("/api/projetos/{projeto_id}", response_model=schemas.Projeto, tags=["Projetos"])
def obter_projeto_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Obtém um projeto específico"""
    db_projeto = crud.obter_projeto(db, projeto_id)
    if not db_projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")
    return db_projeto

@app.get("/api/projetos/{projeto_id}/completo", response_model=schemas.ProjetoCompleto, tags=["Projetos"])
def obter_projeto_completo_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Obtém projeto com todos os dados relacionados"""
    db_projeto = crud.obter_projeto_completo(db, projeto_id)
    if not db_projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")
    return db_projeto

@app.put("/api/projetos/{projeto_id}", response_model=schemas.Projeto, tags=["Projetos"])
def atualizar_projeto_endpoint(projeto_id: str, projeto: schemas.ProjetoUpdate, db: Session = Depends(get_db)):
    """Atualiza um projeto"""
    db_projeto = crud.atualizar_projeto(db, projeto_id, projeto)
    if not db_projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")
    return db_projeto

@app.delete("/api/projetos/{projeto_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["Projetos"])
def deletar_projeto_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Deleta um projeto"""
    success = crud.deletar_projeto(db, projeto_id)
    if not success:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")

# ==================== ROTAS DE ETAPA ====================

@app.post("/api/projetos/{projeto_id}/etapas", response_model=schemas.Etapa, status_code=status.HTTP_201_CREATED, tags=["Etapas"])
def criar_etapa_endpoint(projeto_id: str, etapa: schemas.EtapaBase, db: Session = Depends(get_db)):
    """Cria uma nova etapa"""
    # Verificar se projeto existe
    projeto = crud.obter_projeto(db, projeto_id)
    if not projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")

    etapa_create = schemas.EtapaCreate(**etapa.model_dump(), projeto_id=projeto_id)
    return crud.criar_etapa(db, etapa_create)

@app.get("/api/projetos/{projeto_id}/etapas", response_model=List[schemas.Etapa], tags=["Etapas"])
def listar_etapas_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Lista etapas de um projeto"""
    return crud.listar_etapas(db, projeto_id)

@app.put("/api/etapas/{etapa_id}", response_model=schemas.Etapa, tags=["Etapas"])
def atualizar_etapa_endpoint(etapa_id: str, etapa: schemas.EtapaUpdate, db: Session = Depends(get_db)):
    """Atualiza uma etapa"""
    db_etapa = crud.atualizar_etapa(db, etapa_id, etapa)
    if not db_etapa:
        raise HTTPException(status_code=404, detail="Etapa não encontrada")
    return db_etapa

@app.delete("/api/etapas/{etapa_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["Etapas"])
def deletar_etapa_endpoint(etapa_id: str, db: Session = Depends(get_db)):
    """Deleta uma etapa"""
    success = crud.deletar_etapa(db, etapa_id)
    if not success:
        raise HTTPException(status_code=404, detail="Etapa não encontrada")

# ==================== ROTAS DE ATIVIDADE ====================

@app.post("/api/etapas/{etapa_id}/atividades", response_model=schemas.Atividade, status_code=status.HTTP_201_CREATED, tags=["Atividades"])
def criar_atividade_endpoint(etapa_id: str, atividade: schemas.AtividadeBase, db: Session = Depends(get_db)):
    """Cria uma nova atividade"""
    # Verificar se etapa existe
    etapa = crud.obter_etapa(db, etapa_id)
    if not etapa:
        raise HTTPException(status_code=404, detail="Etapa não encontrada")

    atividade_create = schemas.AtividadeCreate(**atividade.model_dump(), etapa_id=etapa_id)
    return crud.criar_atividade(db, atividade_create)

@app.get("/api/etapas/{etapa_id}/atividades", response_model=List[schemas.Atividade], tags=["Atividades"])
def listar_atividades_endpoint(etapa_id: str, db: Session = Depends(get_db)):
    """Lista atividades de uma etapa"""
    return crud.listar_atividades(db, etapa_id)

@app.put("/api/atividades/{atividade_id}", response_model=schemas.Atividade, tags=["Atividades"])
def atualizar_atividade_endpoint(atividade_id: str, atividade: schemas.AtividadeUpdate, db: Session = Depends(get_db)):
    """Atualiza uma atividade"""
    db_atividade = crud.atualizar_atividade(db, atividade_id, atividade)
    if not db_atividade:
        raise HTTPException(status_code=404, detail="Atividade não encontrada")
    return db_atividade

@app.delete("/api/atividades/{atividade_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["Atividades"])
def deletar_atividade_endpoint(atividade_id: str, db: Session = Depends(get_db)):
    """Deleta uma atividade"""
    success = crud.deletar_atividade(db, atividade_id)
    if not success:
        raise HTTPException(status_code=404, detail="Atividade não encontrada")

# ==================== ROTAS DE DESPESA ====================

@app.post("/api/projetos/{projeto_id}/despesas", response_model=schemas.ItemDespesa, status_code=status.HTTP_201_CREATED, tags=["Despesas"])
def criar_despesa_endpoint(projeto_id: str, despesa: schemas.ItemDespesaBase, db: Session = Depends(get_db)):
    """Cria uma nova despesa"""
    # Verificar se projeto existe
    projeto = crud.obter_projeto(db, projeto_id)
    if not projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")

    despesa_create = schemas.ItemDespesaCreate(**despesa.model_dump(), projeto_id=projeto_id)
    return crud.criar_despesa(db, despesa_create)

@app.get("/api/projetos/{projeto_id}/despesas", response_model=List[schemas.ItemDespesa], tags=["Despesas"])
def listar_despesas_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Lista despesas de um projeto"""
    return crud.listar_despesas(db, projeto_id)

@app.put("/api/despesas/{despesa_id}", response_model=schemas.ItemDespesa, tags=["Despesas"])
def atualizar_despesa_endpoint(despesa_id: str, despesa: schemas.ItemDespesaUpdate, db: Session = Depends(get_db)):
    """Atualiza uma despesa"""
    db_despesa = crud.atualizar_despesa(db, despesa_id, despesa)
    if not db_despesa:
        raise HTTPException(status_code=404, detail="Despesa não encontrada")
    return db_despesa

@app.delete("/api/despesas/{despesa_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["Despesas"])
def deletar_despesa_endpoint(despesa_id: str, db: Session = Depends(get_db)):
    """Deleta uma despesa"""
    success = crud.deletar_despesa(db, despesa_id)
    if not success:
        raise HTTPException(status_code=404, detail="Despesa não encontrada")

# ==================== ROTAS DE FINANCEIRO ====================

@app.get("/api/projetos/{projeto_id}/orcamento-consolidado", tags=["Financeiro"])
def obter_orcamento_consolidado_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Obtém orçamento consolidado do projeto"""
    projeto = crud.obter_projeto(db, projeto_id)
    if not projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")

    return crud.obter_orcamento_consolidado(db, projeto_id)

@app.get("/api/projetos/{projeto_id}/cronograma-financeiro", tags=["Financeiro"])
def obter_cronograma_financeiro_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Obtém cronograma financeiro mensal do projeto"""
    projeto = crud.obter_projeto(db, projeto_id)
    if not projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")

    return crud.obter_cronograma_financeiro(db, projeto_id)

# ==================== ROTAS DE DESEMBOLSO ====================

@app.get("/api/desembolso-regras", response_model=List[schemas.DesembolsoRegra], tags=["Desembolso"])
def listar_desembolso_regras_endpoint(db: Session = Depends(get_db)):
    """Lista todos os modelos de desembolso"""
    return crud.listar_desembolso_regras(db)

@app.post("/api/desembolso-regras", response_model=schemas.DesembolsoRegra, status_code=status.HTTP_201_CREATED, tags=["Desembolso"])
def criar_desembolso_regra_endpoint(regra: schemas.DesembolsoRegraCreate, db: Session = Depends(get_db)):
    """Cria um novo modelo de desembolso"""
    return crud.criar_desembolso_regra(db, regra)

@app.post("/api/projetos/{projeto_id}/aplicar-desembolso", response_model=List[schemas.Desembolso], tags=["Desembolso"])
def aplicar_desembolso_endpoint(projeto_id: str, regra_id: str, db: Session = Depends(get_db)):
    """Aplica modelo de desembolso ao projeto"""
    desembolsos = crud.aplicar_desembolso(db, projeto_id, regra_id)
    if not desembolsos:
        raise HTTPException(status_code=400, detail="Erro ao aplicar desembolso. Verifique se o projeto tem datas definidas.")
    return desembolsos

@app.get("/api/projetos/{projeto_id}/desembolsos", response_model=List[schemas.Desembolso], tags=["Desembolso"])
def listar_desembolsos_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Lista desembolsos de um projeto"""
    return crud.listar_desembolsos(db, projeto_id)

@app.get("/api/projetos/{projeto_id}/comparacao-desembolso", tags=["Desembolso"])
def comparar_desembolso_endpoint(projeto_id: str, db: Session = Depends(get_db)):
    """Compara desembolso com cronograma financeiro"""
    projeto = crud.obter_projeto(db, projeto_id)
    if not projeto:
        raise HTTPException(status_code=404, detail="Projeto não encontrado")

    return crud.comparar_desembolso_cronograma(db, projeto_id)

# ==================== ROTA DE HEALTH CHECK ====================

@app.get("/", tags=["Health"])
def root():
    """Health check endpoint"""
    return {
        "status": "online",
        "message": "API de Gestão de Projetos P&D",
        "version": "1.0.0"
    }

@app.get("/health", tags=["Health"])
def health_check():
    """Verifica saúde da API"""
    return {"status": "healthy"}

Writing backend/main.py


In [8]:
%%writefile backend/init_data.py
# CÉLULA 8: Script para popular dados iniciais

from sqlalchemy.orm import Session
from backend.database import SessionLocal
import backend.models as models
import backend.crud as crud
import backend.schemas as schemas

def criar_modelos_desembolso_padrao():
    """Cria os 3 modelos de desembolso padrão"""
    db = SessionLocal()

    try:
        # Verificar se já existem modelos
        modelos_existentes = crud.listar_desembolso_regras(db)
        if modelos_existentes:
            print("⚠️  Modelos de desembolso já existem")
            return

        # Modelo EMBRAPII
        embrapii = schemas.DesembolsoRegraCreate(
            nome="EMBRAPII Padrão",
            descricao="Modelo padrão EMBRAPII: 3 parcelas (40%, 30%, 30%)",
            tipo=schemas.TipoDesembolsoEnum.EMBRAPII,
            parametros={
                "num_parcelas": 3,
                "distribuicao": [
                    {"parcela": 1, "percentual": 40, "momento": "inicio"},
                    {"parcela": 2, "percentual": 30, "momento": "meio"},
                    {"parcela": 3, "percentual": 30, "momento": "fim"}
                ]
            }
        )
        crud.criar_desembolso_regra(db, embrapii)
        print("✅ Modelo EMBRAPII criado")

        # Modelo SEBRAE
        sebrae = schemas.DesembolsoRegraCreate(
            nome="SEBRAE Trimestral",
            descricao="Modelo SEBRAE: 4 parcelas trimestrais iguais (25% cada)",
            tipo=schemas.TipoDesembolsoEnum.SEBRAE,
            parametros={
                "num_parcelas": 4,
                "distribuicao": [
                    {"parcela": 1, "percentual": 25, "trimestre": 1},
                    {"parcela": 2, "percentual": 25, "trimestre": 2},
                    {"parcela": 3, "percentual": 25, "trimestre": 3},
                    {"parcela": 4, "percentual": 25, "trimestre": 4}
                ]
            }
        )
        crud.criar_desembolso_regra(db, sebrae)
        print("✅ Modelo SEBRAE criado")

        # Modelo BNDES
        bndes = schemas.DesembolsoRegraCreate(
            nome="BNDES Bifásico",
            descricao="Modelo BNDES: 2 parcelas (50% início, 50% fim)",
            tipo=schemas.TipoDesembolsoEnum.BNDES,
            parametros={
                "num_parcelas": 2,
                "distribuicao": [
                    {"parcela": 1, "percentual": 50, "momento": "inicio"},
                    {"parcela": 2, "percentual": 50, "momento": "fim"}
                ]
            }
        )
        crud.criar_desembolso_regra(db, bndes)
        print("✅ Modelo BNDES criado")

        print("\n🎉 Todos os modelos de desembolso foram criados!")

    finally:
        db.close()

if __name__ == "__main__":
    criar_modelos_desembolso_padrao()

Writing backend/init_data.py


In [14]:
# CÉLULA 9 (VERSÃO CORRIGIDA): Iniciar FastAPI no Colab

import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading
import time

# Permitir loops assíncronos aninhados no Colab
nest_asyncio.apply()

print("=" * 60)
print("🚀 INICIANDO SISTEMA")
print("=" * 60)

# 1. Primeiro importar e inicializar o banco
print("\n1️⃣ Inicializando banco de dados...")
from backend.database import init_db
init_db()
print("✅ Tabelas criadas!")

# 2. Importar a aplicação
print("\n2️⃣ Importando aplicação FastAPI...")
from backend.main import app
print("✅ Aplicação carregada!")

# 3. Criar modelos de desembolso padrão
print("\n3️⃣ Criando modelos de desembolso padrão...")
from backend.init_data import criar_modelos_desembolso_padrao
criar_modelos_desembolso_padrao()

# 4. Configurar ngrok
print("\n4️⃣ Configurando acesso público...")
# Se você tiver token do ngrok, descomente e coloque aqui:
# ngrok.set_auth_token("SEU_TOKEN_AQUI")

try:
    public_url = ngrok.connect(8000)
    print("\n" + "=" * 60)
    print("🌐 API DISPONÍVEL PUBLICAMENTE!")
    print("=" * 60)
    print(f"📍 URL Local:      http://127.0.0.1:8000")
    print(f"🌐 URL Pública:    {public_url}")
    print(f"📚 Documentação:   {public_url}/docs")
    print(f"📖 ReDoc:          {public_url}/redoc")
    print("=" * 60)

    # Salvar URL para uso posterior
    import os
    os.environ['API_URL'] = str(public_url)

except Exception as e:
    print(f"\n⚠️ Ngrok não configurado: {e}")
    print("📍 API rodando localmente: http://127.0.0.1:8000")
    print("💡 Para acesso externo, cadastre-se em: https://ngrok.com")
    os.environ['API_URL'] = "http://127.0.0.1:8000"

# 5. Rodar servidor em thread separada
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

print("\n5️⃣ Iniciando servidor...")
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Aguardar servidor iniciar
time.sleep(4)

print("\n" + "=" * 60)
print("✅ SISTEMA RODANDO COM SUCESSO!")
print("=" * 60)
print("\n💡 Próximos passos:")
print("   1. Acesse a documentação em /docs")
print("   2. Execute a célula do Streamlit (próxima)")
print("   3. Ou execute o script de testes")

🚀 INICIANDO SISTEMA

1️⃣ Inicializando banco de dados...
✅ Banco de dados inicializado!
✅ Tabelas criadas!

2️⃣ Importando aplicação FastAPI...
✅ Aplicação carregada!

3️⃣ Criando modelos de desembolso padrão...
✅ Modelo EMBRAPII criado
✅ Modelo SEBRAE criado
✅ Modelo BNDES criado

🎉 Todos os modelos de desembolso foram criados!

4️⃣ Configurando acesso público...


ERROR:pyngrok.process.ngrok:t=2025-10-28T23:04:40+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-10-28T23:04:40+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-10-28T23:04:40+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut


⚠️ Ngrok não configurado: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
📍 API rodando localmente: http://127.0.0.1:8000
💡 Para acesso externo, cadastre-se em: https://ngrok.com

5️⃣ Iniciando servidor...
✅ Banco de dados inicializado!
✅ API inicializada e banco de dados criado!

✅ SISTEMA RODANDO COM SUCESSO!

💡 Próximos passos:
   1. Acesse a documentação em /docs
   2. Execute a célula do Streamlit (próxima)
   3. Ou execute o script de testes


In [15]:
%%writefile frontend/app.py
# CÉLULA 10: Aplicação Streamlit

import streamlit as st
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import date, datetime
from decimal import Decimal

# Configuração da página
st.set_page_config(
    page_title="Gestão de Projetos P&D",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# URL da API (ajuste conforme necessário)
API_URL = "http://127.0.0.1:8000/api"

# Estilo CSS customizado
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        color: #1f77b4;
        text-align: center;
        margin-bottom: 2rem;
    }
    .metric-card {
        background-color: #f0f2f6;
        padding: 1rem;
        border-radius: 0.5rem;
        margin: 0.5rem 0;
    }
    .success-message {
        padding: 1rem;
        background-color: #d4edda;
        border-left: 4px solid #28a745;
        margin: 1rem 0;
    }
    .error-message {
        padding: 1rem;
        background-color: #f8d7da;
        border-left: 4px solid #dc3545;
        margin: 1rem 0;
    }
</style>
""", unsafe_allow_html=True)

# ==================== FUNÇÕES AUXILIARES ====================

def formatar_moeda(valor):
    """Formata valor como moeda brasileira"""
    return f"R$ {float(valor):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def formatar_data(data_str):
    """Formata data no padrão brasileiro"""
    if isinstance(data_str, str):
        data = datetime.strptime(data_str, "%Y-%m-%d").date()
    else:
        data = data_str
    return data.strftime("%d/%m/%Y")

# ==================== FUNÇÕES DE API ====================

def listar_empresas():
    """Lista todas as empresas"""
    try:
        response = requests.get(f"{API_URL}/empresas")
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao listar empresas: {str(e)}")
        return []

def criar_empresa(dados):
    """Cria nova empresa"""
    try:
        response = requests.post(f"{API_URL}/empresas", json=dados)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao criar empresa: {str(e)}")
        return None

def listar_projetos(empresa_id=None):
    """Lista projetos"""
    try:
        url = f"{API_URL}/projetos"
        if empresa_id:
            url += f"?empresa_id={empresa_id}"
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao listar projetos: {str(e)}")
        return []

def obter_projeto_completo(projeto_id):
    """Obtém projeto com todos os dados"""
    try:
        response = requests.get(f"{API_URL}/projetos/{projeto_id}/completo")
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao obter projeto: {str(e)}")
        return None

def criar_projeto(dados):
    """Cria novo projeto"""
    try:
        response = requests.post(f"{API_URL}/projetos", json=dados)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao criar projeto: {str(e)}")
        return None

def criar_etapa(projeto_id, dados):
    """Cria nova etapa"""
    try:
        response = requests.post(f"{API_URL}/projetos/{projeto_id}/etapas", json=dados)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao criar etapa: {str(e)}")
        return None

def criar_atividade(etapa_id, dados):
    """Cria nova atividade"""
    try:
        response = requests.post(f"{API_URL}/etapas/{etapa_id}/atividades", json=dados)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao criar atividade: {str(e)}")
        return None

def criar_despesa(projeto_id, dados):
    """Cria nova despesa"""
    try:
        response = requests.post(f"{API_URL}/projetos/{projeto_id}/despesas", json=dados)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao criar despesa: {str(e)}")
        return None

def obter_orcamento_consolidado(projeto_id):
    """Obtém orçamento consolidado"""
    try:
        response = requests.get(f"{API_URL}/projetos/{projeto_id}/orcamento-consolidado")
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao obter orçamento: {str(e)}")
        return None

def obter_cronograma_financeiro(projeto_id):
    """Obtém cronograma financeiro"""
    try:
        response = requests.get(f"{API_URL}/projetos/{projeto_id}/cronograma-financeiro")
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao obter cronograma financeiro: {str(e)}")
        return None

def listar_modelos_desembolso():
    """Lista modelos de desembolso"""
    try:
        response = requests.get(f"{API_URL}/desembolso-regras")
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao listar modelos: {str(e)}")
        return []

def aplicar_desembolso(projeto_id, regra_id):
    """Aplica modelo de desembolso"""
    try:
        response = requests.post(
            f"{API_URL}/projetos/{projeto_id}/aplicar-desembolso",
            params={"regra_id": regra_id}
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao aplicar desembolso: {str(e)}")
        return None

def listar_desembolsos(projeto_id):
    """Lista desembolsos do projeto"""
    try:
        response = requests.get(f"{API_URL}/projetos/{projeto_id}/desembolsos")
        response.raise_for_status()
        return response.json()
    except Exception as e:
        st.error(f"Erro ao listar desembolsos: {str(e)}")
        return []

# ==================== PÁGINAS ====================

def pagina_dashboard():
    """Dashboard principal"""
    st.markdown('<h1 class="main-header">📊 Dashboard</h1>', unsafe_allow_html=True)

    # Métricas principais
    empresas = listar_empresas()
    projetos = listar_projetos()

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric("Total de Empresas", len(empresas))

    with col2:
        st.metric("Total de Projetos", len(projetos))

    with col3:
        projetos_ativos = [p for p in projetos if p.get('status') == 'ativo']
        st.metric("Projetos Ativos", len(projetos_ativos))

    st.divider()

    # Lista de projetos recentes
    st.subheader("📋 Projetos Recentes")

    if projetos:
        df_projetos = pd.DataFrame(projetos)
        df_projetos['valor_total'] = df_projetos['valor_total'].apply(lambda x: formatar_moeda(x))
        st.dataframe(
            df_projetos[['titulo', 'status', 'valor_total']],
            use_container_width=True,
            hide_index=True
        )
    else:
        st.info("Nenhum projeto cadastrado ainda.")

def pagina_empresas():
    """Página de gestão de empresas"""
    st.markdown('<h1 class="main-header">🏢 Empresas</h1>', unsafe_allow_html=True)

    tab1, tab2 = st.tabs(["📋 Listar", "➕ Nova Empresa"])

    with tab1:
        empresas = listar_empresas()
        if empresas:
            df = pd.DataFrame(empresas)
            st.dataframe(
                df[['nome', 'cnpj', 'email', 'telefone']],
                use_container_width=True,
                hide_index=True
            )
        else:
            st.info("Nenhuma empresa cadastrada.")

    with tab2:
        with st.form("form_empresa"):
            nome = st.text_input("Nome da Empresa *", max_chars=200)
            cnpj = st.text_input("CNPJ * (apenas números)", max_chars=14)
            email = st.text_input("E-mail", max_chars=100)
            telefone = st.text_input("Telefone", max_chars=20)

            submitted = st.form_submit_button("💾 Salvar Empresa")

            if submitted:
                if not nome or not cnpj:
                    st.error("Nome e CNPJ são obrigatórios!")
                elif len(cnpj) != 14 or not cnpj.isdigit():
                    st.error("CNPJ deve ter exatamente 14 dígitos numéricos!")
                else:
                    dados = {
                        "nome": nome,
                        "cnpj": cnpj,
                        "email": email if email else None,
                        "telefone": telefone if telefone else None
                    }
                    resultado = criar_empresa(dados)
                    if resultado:
                        st.success("✅ Empresa criada com sucesso!")
                        st.rerun()

def pagina_projetos():
    """Página de gestão de projetos"""
    st.markdown('<h1 class="main-header">📊 Projetos</h1>', unsafe_allow_html=True)

    tab1, tab2 = st.tabs(["📋 Listar/Gerenciar", "➕ Novo Projeto"])

    with tab1:
        projetos = listar_projetos()
        if projetos:
            # Seletor de projeto
            projeto_selecionado = st.selectbox(
                "Selecione um projeto",
                options=projetos,
                format_func=lambda p: p['titulo']
            )

            if projeto_selecionado:
                projeto_completo = obter_projeto_completo(projeto_selecionado['id'])

                if projeto_completo:
                    # Tabs do projeto
                    tab_dados, tab_cronograma, tab_despesas, tab_desembolso = st.tabs([
                        "📄 Dados", "📅 Cronograma", "💰 Despesas", "💳 Desembolso"
                    ])

                    with tab_dados:
                        col1, col2 = st.columns(2)
                        with col1:
                            st.write("**Título:**", projeto_completo['titulo'])
                            st.write("**Empresa:**", projeto_completo['empresa']['nome'])
                            st.write("**Status:**", projeto_completo['status'])
                        with col2:
                            st.write("**Valor Total:**", formatar_moeda(projeto_completo['valor_total']))
                            if projeto_completo['data_inicio']:
                                st.write("**Início:**", formatar_data(projeto_completo['data_inicio']))
                            if projeto_completo['data_fim']:
                                st.write("**Fim:**", formatar_data(projeto_completo['data_fim']))

                    with tab_cronograma:
                        st.subheader("📅 Etapas e Atividades")

                        # Formulário para nova etapa
                        with st.expander("➕ Adicionar Etapa"):
                            with st.form("form_etapa"):
                                nome_etapa = st.text_input("Nome da Etapa")
                                ordem_etapa = st.number_input("Ordem", min_value=1, value=len(projeto_completo['etapas']) + 1)
                                descricao_etapa = st.text_area("Descrição")

                                if st.form_submit_button("Adicionar Etapa"):
                                    if nome_etapa:
                                        dados = {
                                            "nome": nome_etapa,
                                            "ordem": ordem_etapa,
                                            "descricao": descricao_etapa if descricao_etapa else None
                                        }
                                        resultado = criar_etapa(projeto_completo['id'], dados)
                                        if resultado:
                                            st.success("✅ Etapa criada!")
                                            st.rerun()

                        # Listar etapas e atividades
                        for etapa in projeto_completo['etapas']:
                            with st.expander(f"📌 Etapa {etapa['ordem']}: {etapa['nome']}"):
                                st.write(f"**Descrição:** {etapa.get('descricao', 'N/A')}")

                                # Formulário para nova atividade
                                with st.form(f"form_atividade_{etapa['id']}"):
                                    st.write("**Adicionar Atividade:**")
                                    nome_ativ = st.text_input("Nome", key=f"nome_{etapa['id']}")
                                    col1, col2 = st.columns(2)
                                    with col1:
                                        data_inicio = st.date_input("Data Início", key=f"inicio_{etapa['id']}")
                                    with col2:
                                        data_fim = st.date_input("Data Fim", key=f"fim_{etapa['id']}")
                                    responsavel = st.text_input("Responsável", key=f"resp_{etapa['id']}")

                                    if st.form_submit_button("Adicionar Atividade"):
                                        if nome_ativ and data_inicio and data_fim:
                                            if data_fim < data_inicio:
                                                st.error("Data de fim deve ser >= data de início!")
                                            else:
                                                dados = {
                                                    "nome": nome_ativ,
                                                    "data_inicio": data_inicio.isoformat(),
                                                    "data_fim": data_fim.isoformat(),
                                                    "responsavel": responsavel if responsavel else None,
                                                    "descricao": None
                                                }
                                                resultado = criar_atividade(etapa['id'], dados)
                                                if resultado:
                                                    st.success("✅ Atividade criada!")
                                                    st.rerun()

                                # Listar atividades
                                if etapa['atividades']:
                                    df_ativ = pd.DataFrame(etapa['atividades'])
                                    df_ativ['data_inicio'] = df_ativ['data_inicio'].apply(formatar_data)
                                    df_ativ['data_fim'] = df_ativ['data_fim'].apply(formatar_data)
                                    st.dataframe(
                                        df_ativ[['nome', 'data_inicio', 'data_fim', 'responsavel']],
                                        use_container_width=True,
                                        hide_index=True
                                    )
                                else:
                                    st.info("Nenhuma atividade cadastrada nesta etapa.")

                    with tab_despesas:
                        st.subheader("💰 Despesas do Projeto")

                        # Formulário para nova despesa
                        with st.expander("➕ Adicionar Despesa"):
                            with st.form("form_despesa"):
                                descricao_desp= st.text_input("Descrição *")
                                valor_desp = st.number_input("Valor (R$) *", min_value=0.01, step=0.01)
                                categoria_desp = st.selectbox(
                                    "Categoria *",
                                    ["Pessoal", "Material", "Serviços", "Equipamentos", "Outros"]
                                )

                                st.write("**Percentuais por Fonte (deve somar 100%):**")
                                col1, col2, col3 = st.columns(3)
                                with col1:
                                    perc_embrapii = st.number_input("EMBRAPII (%)", min_value=0.0, max_value=100.0, value=33.33, step=0.01)
                                with col2:
                                    perc_empresa = st.number_input("Empresa (%)", min_value=0.0, max_value=100.0, value=33.33, step=0.01)
                                with col3:
                                    perc_iff = st.number_input("IFF (%)", min_value=0.0, max_value=100.0, value=33.34, step=0.01)

                                soma_perc = perc_embrapii + perc_empresa + perc_iff
                                st.write(f"**Soma:** {soma_perc:.2f}%")

                                if st.form_submit_button("Adicionar Despesa"):
                                    if not descricao_desp:
                                        st.error("Descrição é obrigatória!")
                                    elif abs(soma_perc - 100) > 0.01:
                                        st.error(f"A soma dos percentuais deve ser 100%. Atual: {soma_perc:.2f}%")
                                    else:
                                        dados = {
                                            "descricao": descricao_desp,
                                            "valor": valor_desp,
                                            "categoria": categoria_desp,
                                            "percentual_embrapii": perc_embrapii,
                                            "percentual_empresa": perc_empresa,
                                            "percentual_iff": perc_iff
                                        }
                                        resultado = criar_despesa(projeto_completo['id'], dados)
                                        if resultado:
                                            st.success("✅ Despesa criada!")
                                            st.rerun()

                        # Listar despesas
                        if projeto_completo['despesas']:
                            df_desp = pd.DataFrame(projeto_completo['despesas'])
                            df_desp['valor_fmt'] = df_desp['valor'].apply(formatar_moeda)
                            st.dataframe(
                                df_desp[['descricao', 'categoria', 'valor_fmt', 'percentual_embrapii', 'percentual_empresa', 'percentual_iff']],
                                use_container_width=True,
                                hide_index=True
                            )
                        else:
                            st.info("Nenhuma despesa cadastrada.")

                    with tab_desembolso:
                        st.subheader("💳 Desembolso")

                        modelos = listar_modelos_desembolso()

                        if modelos:
                            modelo_selecionado = st.selectbox(
                                "Selecione o modelo de desembolso",
                                options=modelos,
                                format_func=lambda m: f"{m['nome']} - {m['descricao']}"
                            )

                            if st.button("🚀 Aplicar Modelo ao Projeto"):
                                if not projeto_completo['data_inicio'] or not projeto_completo['data_fim']:
                                    st.error("Projeto precisa ter datas de início e fim definidas. Adicione atividades primeiro!")
                                else:
                                    resultado = aplicar_desembolso(projeto_completo['id'], modelo_selecionado['id'])
                                    if resultado:
                                        st.success("✅ Desembolso aplicado com sucesso!")
                                        st.rerun()

                        st.divider()

                        # Listar desembolsos aplicados
                        desembolsos = listar_desembolsos(projeto_completo['id'])

                        if desembolsos:
                            st.write("**Parcelas de Desembolso:**")
                            df_desemb = pd.DataFrame(desembolsos)
                            df_desemb['valor_fmt'] = df_desemb['valor'].apply(formatar_moeda)
                            df_desemb['data_prevista'] = df_desemb['data_prevista'].apply(formatar_data)
                            st.dataframe(
                                df_desemb[['parcela', 'valor_fmt', 'percentual', 'data_prevista']],
                                use_container_width=True,
                                hide_index=True
                            )

                            total_desembolso = sum(float(d['valor']) for d in desembolsos)
                            st.metric("Total do Desembolso", formatar_moeda(total_desembolso))
                        else:
                            st.info("Nenhum modelo de desembolso aplicado ainda.")
        else:
            st.info("Nenhum projeto cadastrado ainda.")

    with tab2:
        st.subheader("➕ Criar Novo Projeto")

        empresas = listar_empresas()

        if not empresas:
            st.warning("⚠️ Cadastre uma empresa primeiro!")
        else:
            with st.form("form_projeto"):
                empresa_sel = st.selectbox(
                    "Empresa *",
                    options=empresas,
                    format_func=lambda e: e['nome']
                )

                titulo = st.text_input("Título do Projeto *", max_chars=300)
                descricao = st.text_area("Descrição")

                col1, col2 = st.columns(2)
                with col1:
                    status = st.selectbox("Status", ["ativo", "concluido", "cancelado"])
                with col2:
                    valor_total = st.number_input("Valor Total (R$)", min_value=0.0, value=0.0, step=1000.0)

                submitted = st.form_submit_button("💾 Criar Projeto")

                if submitted:
                    if not titulo:
                        st.error("Título é obrigatório!")
                    else:
                        dados = {
                            "empresa_id": empresa_sel['id'],
                            "titulo": titulo,
                            "descricao": descricao if descricao else None,
                            "status": status,
                            "valor_total": valor_total,
                            "data_inicio": None,
                            "data_fim": None
                        }
                        resultado = criar_projeto(dados)
                        if resultado:
                            st.success("✅ Projeto criado com sucesso!")
                            st.info("💡 Agora adicione etapas e atividades na aba 'Listar/Gerenciar'")
                            st.rerun()

def pagina_financeiro():
    """Página de análise financeira"""
    st.markdown('<h1 class="main-header">💰 Análise Financeira</h1>', unsafe_allow_html=True)

    projetos = listar_projetos()

    if not projetos:
        st.info("Nenhum projeto cadastrado ainda.")
        return

    projeto_sel = st.selectbox(
        "Selecione um projeto",
        options=projetos,
        format_func=lambda p: p['titulo']
    )

    if projeto_sel:
        projeto_id = projeto_sel['id']

        # Orçamento Consolidado
        st.subheader("📊 Orçamento Consolidado")

        orcamento = obter_orcamento_consolidado(projeto_id)

        if orcamento and orcamento['total_geral'] > 0:
            # Métricas principais
            col1, col2, col3, col4 = st.columns(4)

            with col1:
                st.metric("Total Geral", formatar_moeda(orcamento['total_geral']))
            with col2:
                st.metric("EMBRAPII", formatar_moeda(orcamento['por_fonte']['EMBRAPII']))
            with col3:
                st.metric("Empresa", formatar_moeda(orcamento['por_fonte']['Empresa']))
            with col4:
                st.metric("IFF", formatar_moeda(orcamento['por_fonte']['IFF']))

            st.divider()

            # Gráficos
            col1, col2 = st.columns(2)

            with col1:
                st.subheader("💰 Por Fonte Financiadora")
                fig_fonte = px.pie(
                    values=list(orcamento['por_fonte'].values()),
                    names=list(orcamento['por_fonte'].keys()),
                    title="Distribuição por Fonte",
                    hole=0.4
                )
                st.plotly_chart(fig_fonte, use_container_width=True)

            with col2:
                st.subheader("📦 Por Categoria")
                fig_cat = px.bar(
                    x=list(orcamento['por_categoria'].keys()),
                    y=list(orcamento['por_categoria'].values()),
                    title="Despesas por Categoria",
                    labels={'x': 'Categoria', 'y': 'Valor (R$)'}
                )
                st.plotly_chart(fig_cat, use_container_width=True)

            st.divider()

            # Tabelas
            col1, col2 = st.columns(2)

            with col1:
                st.write("**Consolidado por Fonte:**")
                df_fonte = pd.DataFrame([
                    {"Fonte": k, "Valor": formatar_moeda(v)}
                    for k, v in orcamento['por_fonte'].items()
                ])
                st.dataframe(df_fonte, use_container_width=True, hide_index=True)

            with col2:
                st.write("**Consolidado por Categoria:**")
                df_cat = pd.DataFrame([
                    {"Categoria": k, "Valor": formatar_moeda(v)}
                    for k, v in orcamento['por_categoria'].items()
                ])
                st.dataframe(df_cat, use_container_width=True, hide_index=True)
        else:
            st.info("Nenhuma despesa cadastrada para este projeto.")

        st.divider()

        # Cronograma Financeiro
        st.subheader("📅 Cronograma Financeiro Mensal")

        cronograma = obter_cronograma_financeiro(projeto_id)

        if cronograma and cronograma['meses']:
            # Gráfico de linha - Fluxo de Caixa
            fig_fluxo = go.Figure()

            fig_fluxo.add_trace(go.Scatter(
                x=cronograma['meses'],
                y=cronograma['valores'],
                mode='lines+markers',
                name='Mensal',
                line=dict(color='#1f77b4', width=2)
            ))

            fig_fluxo.add_trace(go.Scatter(
                x=cronograma['meses'],
                y=cronograma['acumulado'],
                mode='lines+markers',
                name='Acumulado',
                line=dict(color='#2ca02c', width=2)
            ))

            fig_fluxo.update_layout(
                title="Fluxo de Caixa",
                xaxis_title="Mês",
                yaxis_title="Valor (R$)",
                hovermode='x unified'
            )

            st.plotly_chart(fig_fluxo, use_container_width=True)

            # Tabela do cronograma
            df_cronograma = pd.DataFrame({
                'Mês': cronograma['meses'],
                'Valor Mensal': [formatar_moeda(v) for v in cronograma['valores']],
                'Acumulado': [formatar_moeda(v) for v in cronograma['acumulado']]
            })

            st.dataframe(df_cronograma, use_container_width=True, hide_index=True)
        else:
            st.info("Adicione atividades com datas ao projeto para visualizar o cronograma financeiro.")

def pagina_documentos():
    """Página de geração de documentos"""
    st.markdown('<h1 class="main-header">📄 Documentos</h1>', unsafe_allow_html=True)

    st.info("🚧 Módulo de geração de documentos em desenvolvimento...")
    st.write("Em breve você poderá gerar:")
    st.write("- 📝 Plano de Trabalho (DOCX/PDF)")
    st.write("- 📋 Acordo de Trabalho (DOCX/PDF)")
    st.write("- 📊 Cronograma (Excel)")
    st.write("- 💰 Orçamento (Excel)")

# ==================== NAVEGAÇÃO ====================

def main():
    """Função principal do app"""

    # Sidebar
    st.sidebar.title("🎯 Navegação")

    pagina = st.sidebar.radio(
        "Escolha uma página:",
        ["🏠 Dashboard", "🏢 Empresas", "📊 Projetos", "💰 Financeiro", "📄 Documentos"]
    )

    st.sidebar.divider()

    st.sidebar.info("""
    **Sistema de Gestão de Projetos P&D**

    Versão 1.0.0

    Desenvolvido para gerenciar projetos de pesquisa e desenvolvimento.
    """)

    # Roteamento
    if pagina == "🏠 Dashboard":
        pagina_dashboard()
    elif pagina == "🏢 Empresas":
        pagina_empresas()
    elif pagina == "📊 Projetos":
        pagina_projetos()
    elif pagina == "💰 Financeiro":
        pagina_financeiro()
    elif pagina == "📄 Documentos":
        pagina_documentos()

if __name__ == "__main__":
    main()

Writing frontend/app.py


In [17]:
# CÉLULA: Usar Cloudflare Quick Tunnel (SEM CADASTRO!)

import nest_asyncio
import uvicorn
import threading
import time
import subprocess
import re

nest_asyncio.apply()

# 1. Baixar cloudflared
print("📦 Instalando cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Inicializar banco de dados
print("🔧 Inicializando banco de dados...")
from backend.database import init_db
init_db()

# 3. Importar app
print("📦 Carregando aplicação...")
from backend.main import app

# 4. Criar modelos de desembolso
print("💾 Criando modelos de desembolso...")
from backend.init_data import criar_modelos_desembolso_padrao
criar_modelos_desembolso_padrao()

# 5. Iniciar servidor FastAPI
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

print("🚀 Iniciando servidor...")
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(3)

# 6. Criar túnel com cloudflared
print("🌐 Criando túnel público (pode levar ~10 segundos)...")

tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Aguardar e capturar URL
url = None
for _ in range(30):  # Tentar por 30 segundos
    line = tunnel_process.stderr.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
    time.sleep(1)

if url:
    print("\n" + "=" * 70)
    print("✅ API RODANDO COM SUCESSO!")
    print("=" * 70)
    print(f"🌐 URL Pública:    {url}")
    print(f"📚 Documentação:   {url}/docs")
    print(f"📖 ReDoc:          {url}/redoc")
    print("=" * 70)
    print("\n💡 Use esta URL para acessar a API!")
    print("💡 Esta URL é temporária e muda a cada execução")
    print("⚠️ Mantenha esta célula rodando enquanto usar o sistema")

    # Salvar URL em variável de ambiente
    import os
    os.environ['API_URL'] = url
else:
    print("❌ Erro ao criar túnel. Tente executar a célula novamente.")

📦 Instalando cloudflared...
🔧 Inicializando banco de dados...
✅ Banco de dados inicializado!
📦 Carregando aplicação...
💾 Criando modelos de desembolso...
⚠️  Modelos de desembolso já existem
🚀 Iniciando servidor...


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use


✅ Banco de dados inicializado!
✅ API inicializada e banco de dados criado!
🌐 Criando túnel público (pode levar ~10 segundos)...

✅ API RODANDO COM SUCESSO!
🌐 URL Pública:    https://leaves-occasional-retain-changed.trycloudflare.com
📚 Documentação:   https://leaves-occasional-retain-changed.trycloudflare.com/docs
📖 ReDoc:          https://leaves-occasional-retain-changed.trycloudflare.com/redoc

💡 Use esta URL para acessar a API!
💡 Esta URL é temporária e muda a cada execução
⚠️ Mantenha esta célula rodando enquanto usar o sistema


In [20]:
# CÉLULA: Rodar Streamlit (CORRIGIDA)

import os
import time
import subprocess
import re

# IMPORTANTE: Cole aqui a URL da sua API (a que foi gerada na célula anterior)
API_URL_PUBLICA = "https://leaves-occasional-retain-changed.trycloudflare.com"

print(f"🔧 Configurando Streamlit para usar API: {API_URL_PUBLICA}")

# Ler o arquivo app.py atual
with open('frontend/app.py', 'r') as f:
    content = f.read()

# Substituir a URL da API
content = content.replace(
    'API_URL = "http://127.0.0.1:8000/api"',
    f'API_URL = "{API_URL_PUBLICA}/api"'
)

# Salvar arquivo atualizado
with open('frontend/app.py', 'w') as f:
    f.write(content)

print("✅ Arquivo app.py atualizado com a URL da API!")

# Matar processos streamlit anteriores
print("🔄 Encerrando processos anteriores...")
!pkill -f streamlit
time.sleep(2)

# Rodar Streamlit em background
print("🚀 Iniciando Streamlit...")
!nohup streamlit run frontend/app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

time.sleep(5)

# Criar túnel para Streamlit
print("🌐 Criando túnel público para Streamlit...")

streamlit_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Aguardar URL do túnel
streamlit_url = None
print("⏳ Aguardando túnel (pode levar ~10 segundos)...")

for attempt in range(30):
    line = streamlit_process.stderr.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            streamlit_url = match.group(0)
            break
    time.sleep(1)

print("\n" + "=" * 70)
if streamlit_url:
    print("✅ STREAMLIT RODANDO COM SUCESSO!")
    print("=" * 70)
    print(f"🎨 Interface Web:  {streamlit_url}")
    print(f"📊 API Backend:    {API_URL_PUBLICA}")
    print("=" * 70)
    print("\n💡 Acesse a URL acima no seu navegador!")
    print("💡 Todas as operações serão salvas no banco de dados")
    print("⚠️ Mantenha esta célula rodando enquanto usar o sistema")
else:
    print("❌ ERRO ao criar túnel para Streamlit")
    print("=" * 70)
    print("\n🔧 Tente executar novamente ou use este comando manual:")
    print("!cloudflared tunnel --url http://localhost:8501")

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-4' coro=<Server.serve() done, defined at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:69> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 164, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/base_events.py", line 1584, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-3251462152.py", lin

🔧 Configurando Streamlit para usar API: https://leaves-occasional-retain-changed.trycloudflare.com
✅ Arquivo app.py atualizado com a URL da API!
🔄 Encerrando processos anteriores...
🚀 Iniciando Streamlit...
🌐 Criando túnel público para Streamlit...
⏳ Aguardando túnel (pode levar ~10 segundos)...

✅ STREAMLIT RODANDO COM SUCESSO!
🎨 Interface Web:  https://recommend-yeah-constantly-jill.trycloudflare.com
📊 API Backend:    https://leaves-occasional-retain-changed.trycloudflare.com

💡 Acesse a URL acima no seu navegador!
💡 Todas as operações serão salvas no banco de dados
⚠️ Mantenha esta célula rodando enquanto usar o sistema


In [21]:
%%writefile requirements.txt
# CÉLULA 12: Dependências do Projeto

# Backend
fastapi==0.104.1
uvicorn[standard]==0.24.0
sqlalchemy==2.0.23
pydantic==2.5.0
python-multipart==0.0.6

# Frontend
streamlit==1.28.2
plotly==5.18.0
pandas==2.1.3
openpyxl==3.1.2

# Documentos
python-docx==1.1.0
Jinja2==3.1.2
weasyprint==60.1

# Utilitários
requests==2.31.0
python-dateutil==2.8.2

# Para Colab
pyngrok==7.0.1
nest-asyncio==1.5.8

Writing requirements.txt


In [23]:
# CÉLULA 13: Script de Teste do Sistema

import requests
import json
from datetime import date, timedelta

API_URL = "http://127.0.0.1:8000/api"

def testar_sistema_completo():
    """Testa todas as funcionalidades do sistema"""

    print("🧪 INICIANDO TESTES DO SISTEMA")
    print("=" * 60)

    # 1. Criar Empresa
    print("\n1️⃣ Criando empresa...")
    empresa_data = {
        "nome": "Tech Innovation LTDA",
        "cnpj": "12345678901234",
        "email": "contato@techinnovation.com",
        "telefone": "11999998888"
    }

    response = requests.post(f"{API_URL}/empresas", json=empresa_data)
    if response.status_code == 201:
        empresa = response.json()
        print(f"✅ Empresa criada: {empresa['nome']} (ID: {empresa['id']})")
    else:
        print(f"❌ Erro ao criar empresa: {response.text}")
        return

    # 2. Criar Projeto
    print("\n2️⃣ Criando projeto...")
    projeto_data = {
        "empresa_id": empresa['id'],
        "titulo": "Desenvolvimento de Sistema IoT",
        "descricao": "Sistema de monitoramento inteligente",
        "status": "ativo",
        "valor_total": 0
    }

    response = requests.post(f"{API_URL}/projetos", json=projeto_data)
    if response.status_code == 201:
        projeto = response.json()
        print(f"✅ Projeto criado: {projeto['titulo']} (ID: {projeto['id']})")
    else:
        print(f"❌ Erro ao criar projeto: {response.text}")
        return

    # 3. Criar Etapa
    print("\n3️⃣ Criando etapa...")
    etapa_data = {
        "nome": "Fase de Planejamento",
        "descricao": "Levantamento de requisitos e planejamento",
        "ordem": 1
    }

    response = requests.post(f"{API_URL}/projetos/{projeto['id']}/etapas", json=etapa_data)
    if response.status_code == 201:
        etapa = response.json()
        print(f"✅ Etapa criada: {etapa['nome']} (ID: {etapa['id']})")
    else:
        print(f"❌ Erro ao criar etapa: {response.text}")
        return

    # 4. Criar Atividade
    print("\n4️⃣ Criando atividade...")
    hoje = date.today()
    atividade_data = {
        "nome": "Levantamento de Requisitos",
        "descricao": "Análise completa dos requisitos do sistema",
        "data_inicio": hoje.isoformat(),
        "data_fim": (hoje + timedelta(days=30)).isoformat(),
        "responsavel": "João Silva"
    }

    response = requests.post(f"{API_URL}/etapas/{etapa['id']}/atividades", json=atividade_data)
    if response.status_code == 201:
        atividade = response.json()
        print(f"✅ Atividade criada: {atividade['nome']}")
    else:
        print(f"❌ Erro ao criar atividade: {response.text}")
        return

    # 5. Criar Despesas
    print("\n5️⃣ Criando despesas...")
    despesas = [
        {
            "descricao": "Salário Desenvolvedor Sênior",
            "valor": 15000.00,
            "categoria": "Pessoal",
            "percentual_embrapii": 40.0,
            "percentual_empresa": 30.0,
            "percentual_iff": 30.0
        },
        {
            "descricao": "Equipamento Servidor",
            "valor": 8000.00,
            "categoria": "Equipamentos",
            "percentual_embrapii": 50.0,
            "percentual_empresa": 50.0,
            "percentual_iff": 0.0
        },
        {
            "descricao": "Material de Escritório",
            "valor": 2000.00,
            "categoria": "Material",
            "percentual_embrapii": 33.33,
            "percentual_empresa": 33.33,
            "percentual_iff": 33.34
        }
    ]

    for desp in despesas:
        response = requests.post(f"{API_URL}/projetos/{projeto['id']}/despesas", json=desp)
        if response.status_code == 201:
            print(f"✅ Despesa criada: {desp['descricao']}")
        else:
            print(f"❌ Erro ao criar despesa: {response.text}")

    # 6. Obter Orçamento Consolidado
    print("\n6️⃣ Obtendo orçamento consolidado...")
    response = requests.get(f"{API_URL}/projetos/{projeto['id']}/orcamento-consolidado")
    if response.status_code == 200:
        orcamento = response.json()
        print(f"✅ Total Geral: R$ {orcamento['total_geral']:,.2f}")
        print(f"   - EMBRAPII: R$ {orcamento['por_fonte']['EMBRAPII']:,.2f}")
        print(f"   - Empresa: R$ {orcamento['por_fonte']['Empresa']:,.2f}")
        print(f"   - IFF: R$ {orcamento['por_fonte']['IFF']:,.2f}")
    else:
        print(f"❌ Erro ao obter orçamento: {response.text}")

    # 7. Listar Modelos de Desembolso
    print("\n7️⃣ Listando modelos de desembolso...")
    response = requests.get(f"{API_URL}/desembolso-regras")
    if response.status_code == 200:
        modelos = response.json()
        for modelo in modelos:
            print(f"✅ Modelo: {modelo['nome']}")

        # 8. Aplicar Modelo EMBRAPII
        if modelos:
            print("\n8️⃣ Aplicando modelo EMBRAPII ao projeto...")
            modelo_embrapii = next((m for m in modelos if m['tipo'] == 'embrapii'), None)
            if modelo_embrapii:
                response = requests.post(
                    f"{API_URL}/projetos/{projeto['id']}/aplicar-desembolso",
                    params={"regra_id": modelo_embrapii['id']}
                )
                if response.status_code == 200:
                    desembolsos = response.json()
                    print(f"✅ Desembolso aplicado: {len(desembolsos)} parcelas criadas")
                    for d in desembolsos:
                        print(f"   - Parcela {d['parcela']}: R$ {d['valor']:,.2f} ({d['percentual']}%)")
                else:
                    print(f"❌ Erro ao aplicar desembolso: {response.text}")
    else:
        print(f"❌ Erro ao listar modelos: {response.text}")

    # 9. Obter Projeto Completo
    print("\n9️⃣ Obtendo projeto completo...")
    response = requests.get(f"{API_URL}/projetos/{projeto['id']}/completo")
    if response.status_code == 200:
        projeto_completo = response.json()
        print(f"✅ Projeto: {projeto_completo['titulo']}")
        print(f"   - Empresa: {projeto_completo['empresa']['nome']}")
        print(f"   - Etapas: {len(projeto_completo['etapas'])}")
        print(f"   - Despesas: {len(projeto_completo['despesas'])}")
        print(f"   - Desembolsos: {len(projeto_completo['desembolsos'])}")
    else:
        print(f"❌ Erro ao obter projeto: {response.text}")

    print("\n" + "=" * 60)
    print("🎉 TESTES CONCLUÍDOS COM SUCESSO!")
    print("=" * 60)

# Executar testes
testar_sistema_completo()

🧪 INICIANDO TESTES DO SISTEMA

1️⃣ Criando empresa...
✅ Empresa criada: Tech Innovation LTDA (ID: bbdaee0a-92e0-4b4d-9a3c-17e87052ee58)

2️⃣ Criando projeto...
✅ Projeto criado: Desenvolvimento de Sistema IoT (ID: e34c7008-96b7-42e5-8873-0cc13145d765)

3️⃣ Criando etapa...
✅ Etapa criada: Fase de Planejamento (ID: 12c8af16-2695-4033-ab82-dae5a2b9aeed)

4️⃣ Criando atividade...
✅ Atividade criada: Levantamento de Requisitos

5️⃣ Criando despesas...
✅ Despesa criada: Salário Desenvolvedor Sênior
✅ Despesa criada: Equipamento Servidor
✅ Despesa criada: Material de Escritório

6️⃣ Obtendo orçamento consolidado...
✅ Total Geral: R$ 25,000.00
   - EMBRAPII: R$ 10,666.60
   - Empresa: R$ 9,166.60
   - IFF: R$ 5,166.80

7️⃣ Listando modelos de desembolso...
✅ Modelo: EMBRAPII Padrão
✅ Modelo: SEBRAE Trimestral
✅ Modelo: BNDES Bifásico

8️⃣ Aplicando modelo EMBRAPII ao projeto...
✅ Desembolso aplicado: 3 parcelas criadas


ValueError: Unknown format code 'f' for object of type 'str'

In [22]:
# CÉLULA 14: Verificação Final

print("📋 CHECKLIST DO MVP")
print("=" * 60)

checklist = {
    "✅ Backend - Models": "Todas as entidades criadas",
    "✅ Backend - Schemas": "Validações Pydantic implementadas",
    "✅ Backend - CRUD": "Operações de banco funcionando",
    "✅ Backend - API": "FastAPI com endpoints completos",
    "✅ Frontend - Streamlit": "Interface funcional",
    "✅ Financeiro - Orçamento": "Consolidado por categoria e fonte",
    "✅ Financeiro - Cronograma": "Distribuição mensal implementada",
    "✅ Desembolso - Modelos": "3 modelos pré-configurados",
    "✅ Desembolso - Aplicação": "Motor de regras funcionando",
    "⚠️ Documentos - Templates": "Em desenvolvimento (Fase 2)",
    "⚠️ Documentos - Export": "Em desenvolvimento (Fase 2)"
}

for item, status in checklist.items():
    print(f"{item}: {status}")

print("=" * 60)
print("\n🚀 SISTEMA PRONTO PARA USO!")
print("\n📚 Próximos passos:")
print("1. Acesse a documentação da API: {URL_PUBLICA}/docs")
print("2. Acesse a interface Streamlit: {STREAMLIT_URL}")
print("3. Teste criando empresa → projeto → etapas → despesas")
print("4. Visualize os gráficos financeiros")
print("5. Aplique modelos de desembolso")

📋 CHECKLIST DO MVP
✅ Backend - Models: Todas as entidades criadas
✅ Backend - Schemas: Validações Pydantic implementadas
✅ Backend - CRUD: Operações de banco funcionando
✅ Backend - API: FastAPI com endpoints completos
✅ Frontend - Streamlit: Interface funcional
✅ Financeiro - Orçamento: Consolidado por categoria e fonte
✅ Financeiro - Cronograma: Distribuição mensal implementada
✅ Desembolso - Modelos: 3 modelos pré-configurados
✅ Desembolso - Aplicação: Motor de regras funcionando
⚠️ Documentos - Templates: Em desenvolvimento (Fase 2)
⚠️ Documentos - Export: Em desenvolvimento (Fase 2)

🚀 SISTEMA PRONTO PARA USO!

📚 Próximos passos:
1. Acesse a documentação da API: {URL_PUBLICA}/docs
2. Acesse a interface Streamlit: {STREAMLIT_URL}
3. Teste criando empresa → projeto → etapas → despesas
4. Visualize os gráficos financeiros
5. Aplique modelos de desembolso
